### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [ ]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

In [ ]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    director: str = Field(description="The director of the movie")
    release_year: int = Field(description="The year the movie was released")
    rating: float = Field(description="The rating of the movie out of 10")


model_with_structure = model.with_structured_output(Movie)
model_with_structure

#default output:
model.invoke("What is the movie Inception about?")


#structured output:model_with_structured_output.invoke("What is the movie Inception about?")

response = model_with_structure.invoke("What is the movie Inception about?")
print(response)

### Message output with the parsed structure:

In [ ]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    """A movie with a title, director, release year, and rating."""
    title: str = Field(...,description="The title of the movie")
    director: str = Field(...,description="The director of the movie")
    release_year: int = Field(...,description="The year the movie was released")
    rating: float = Field(...,description="The rating of the movie out of 10")


model_with_structure = model.with_structured_output(Movie, include_raw=True)


#structured output:model_with_structured_output.invoke("What is the movie Inception about?")

response = model_with_structure.invoke("What is the movie Inception about?")
print(response)

### Nested Structure:


In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str 
    role: str

class MovieDetails(BaseModel):
    title: str 
    director: str 
    release_year: int 
    actors: list[Actor]
    genre: list[str]
    budget: float | None = Field(default=None, description="The budget of the movie in millions USD")

model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)

response = model_with_structure.invoke("What is the movie Inception about?")
print(response)

### TypedDict:
 It provides a simpler alternate using Python's buit-in typing, here we don't need run time validation.

In [ ]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str,..., "The title of the movie"]
    year: Annotated[int,..., "The release year of the movie"]
    director: Annotated[str,..., "The director of the movie"]
    rating: Annotated[float,..., "The rating of the movie out of 10"]


model_with_typeddict = model.with_structured_output(MovieDict, include_raw=True)
response = model_with_typeddict.invoke("What is the movie Inception about?")
print(response)

In [ ]:
#Nested TypedDicts:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    director: str
    release_year: int
    actors: list[Actor]
    genre: list[str]
    budget: float | None = Field(default=None, description="The budget of the movie in millions USD")

model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)
response = model_with_structure.invoke("What is the movie Inception about?")
print(response)

### DataClass
A data class is a class typically conatining data, although there are not any restrictions. We create it using @dataclass decorator.

In [ ]:
#with pydantic schema:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """A contact info for a person"""
    name: str = Field(..., description="The name of the contact")
    email: str = Field(..., description="The email address of the contact")
    phone: str = Field(..., description="The phone number of the contact")

agent = create_agent(
    model = "gpt-5",
    response_format=ContactInfo,  #Auto select structured output format
)

response = agent.invoke({
    "messages": [{"role": "user", "content": "Please extract contact info from: John Doe, john@example.com,(555) 123-4656"}]
})

print(response["structured_response"])

In [ ]:
#with typedDict schema:
from typing_extensions import TypedDict, Annotated
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """A contact info for a person"""
    name: str
    email: str
    phone: str

agent = create_agent(
    model = "gpt-5",
    response_format=ContactInfo,  #Auto select structured output format
)

response = agent.invoke({
    "messages": [{"role": "user", "content": "Please extract contact info from: John Doe, john@example.com,(555) 123-4656"}]
})

print(response["structured_response"])

In [ ]:
#with DataClass schema:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """A contact info for a person"""
    name: str
    email: str
    phone: str

agent = create_agent(
    model = "gpt-5",
    response_format=ContactInfo,  #Auto select structured output format
)

response = agent.invoke({
    "messages": [{"role": "user", "content": "Please extract contact info from: John Doe, john@example.com,(555) 123-4656"}]
})

print(response["structured_response"])
